# ПЗ 4 — Извлечение аудио и распознавание речи (Whisper)

Вытаскиваем аудиодорожку из видео через moviepy, прогоняем через Whisper, получаем текст с таймкодами.

In [ ]:
!pip install moviepy openai-whisper -q
!apt-get install -y ffmpeg -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
videos = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.avi', '.mkv'))]
print(f'видео в drive: {videos}')
video_path = f'{VIDEO_DIR}/{videos[0]}'

In [ ]:
from moviepy.editor import VideoFileClip

audio_path = '/content/audio.wav'
clip = VideoFileClip(video_path)
clip.audio.write_audiofile(audio_path, verbose=False, logger=None)
clip.close()
print('аудио извлечено')

In [ ]:
import whisper

model = whisper.load_model('base')
result = model.transcribe(audio_path, language='ru', verbose=False)

print('транскрипция готова')
print(result['text'][:500])

In [ ]:
import pandas as pd

rows = [
    {'start': round(s['start'], 2), 'end': round(s['end'], 2), 'text': s['text'].strip()}
    for s in result['segments']
]

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/whisper_transcript.csv', index=False, encoding='utf-8-sig')
print(f'сегментов: {len(df)}')
df.head(10)